In [ ]:
!pip install -q datasets

In [ ]:
import json

import pandas as pd
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import CachedMultipleNegativesSymmetricRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import InformationRetrievalEvaluator


In [ ]:
CSV_PATH = 'pvs-pairs.csv'

CORPUS_IDS = 'pvs-lemmas.json'
CORPUS_PATH = 'pvs-lemmas-scopes.json'


df = pd.read_csv(CSV_PATH).fillna('')

In [ ]:
lemmas_ids = json.load(open(CORPUS_IDS))
lemmas_scopes = json.load(open(CORPUS_PATH))

In [ ]:
ds = Dataset.from_pandas(df).train_test_split(test_size=.1)
ds_ = ds.remove_columns(["positive_id", "lemma_name"])

ds

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'positive_id', 'lemma_name'],
        num_rows: 785
    })
    test: Dataset({
        features: ['anchor', 'positive', 'positive_id', 'lemma_name'],
        num_rows: 88
    })
})

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/all-distilroberta-v1"
)

loss = CachedMultipleNegativesSymmetricRankingLoss(model)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
queries = {int(id): sequent for id, sequent in enumerate(ds["test"]["anchor"])}
corpus = {int(id): scope for id, scope in lemmas_scopes.items()}
relevant_docs = {qid: {lemmas_ids.get(sample["lemma_name"], 0)}
                 for qid, sample in enumerate(ds["test"])}


dev_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="lemma-retreiever",
)

dev_evaluator(model)

{'lemma-retreiever_cosine_accuracy@1': 0.17045454545454544,
 'lemma-retreiever_cosine_accuracy@3': 0.2727272727272727,
 'lemma-retreiever_cosine_accuracy@5': 0.3409090909090909,
 'lemma-retreiever_cosine_accuracy@10': 0.375,
 'lemma-retreiever_cosine_precision@1': 0.17045454545454544,
 'lemma-retreiever_cosine_precision@3': 0.0909090909090909,
 'lemma-retreiever_cosine_precision@5': 0.06818181818181818,
 'lemma-retreiever_cosine_precision@10': 0.0375,
 'lemma-retreiever_cosine_recall@1': 0.17045454545454544,
 'lemma-retreiever_cosine_recall@3': 0.2727272727272727,
 'lemma-retreiever_cosine_recall@5': 0.3409090909090909,
 'lemma-retreiever_cosine_recall@10': 0.375,
 'lemma-retreiever_cosine_ndcg@10': 0.2730155162946111,
 'lemma-retreiever_cosine_mrr@10': 0.2399621212121212,
 'lemma-retreiever_cosine_map@100': 0.2452037163487365}

In [ ]:
args = SentenceTransformerTrainingArguments(
    output_dir="result",
    num_train_epochs=15,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    warmup_ratio=0.1,
    bf16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=2,
    logging_steps=10,
    report_to="none"
)


trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=ds_["train"],
    eval_dataset=ds_["test"],
    loss=loss,
    evaluator=dev_evaluator,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [ ]:
info = trainer.train()
info

Step,Training Loss,Validation Loss,Lemma-retreiever Cosine Accuracy@1,Lemma-retreiever Cosine Accuracy@3,Lemma-retreiever Cosine Accuracy@5,Lemma-retreiever Cosine Accuracy@10,Lemma-retreiever Cosine Precision@1,Lemma-retreiever Cosine Precision@3,Lemma-retreiever Cosine Precision@5,Lemma-retreiever Cosine Precision@10,Lemma-retreiever Cosine Recall@1,Lemma-retreiever Cosine Recall@3,Lemma-retreiever Cosine Recall@5,Lemma-retreiever Cosine Recall@10,Lemma-retreiever Cosine Ndcg@10,Lemma-retreiever Cosine Mrr@10,Lemma-retreiever Cosine Map@100
10,1.583700,0.394133,0.329545,0.522727,0.625000,0.727273,0.329545,0.174242,0.125000,0.072727,0.329545,0.522727,0.625000,0.727273,0.517682,0.451953,0.459065
20,0.689000,0.313656,0.397727,0.590909,0.738636,0.840909,0.397727,0.196970,0.147727,0.084091,0.397727,0.590909,0.738636,0.840909,0.606559,0.532639,0.538216
30,0.551500,0.326092,0.386364,0.568182,0.670455,0.795455,0.386364,0.189394,0.134091,0.079545,0.386364,0.568182,0.670455,0.795455,0.577395,0.508784,0.515900
40,0.366600,0.205467,0.511364,0.681818,0.772727,0.840909,0.511364,0.227273,0.154545,0.084091,0.511364,0.681818,0.772727,0.840909,0.670944,0.616599,0.622831
50,0.288000,0.320111,0.477273,0.659091,0.761364,0.829545,0.477273,0.219697,0.152273,0.082955,0.477273,0.659091,0.761364,0.829545,0.647562,0.589737,0.596388
60,0.264200,0.293230,0.488636,0.670455,0.795455,0.852273,0.488636,0.223485,0.159091,0.085227,0.488636,0.670455,0.795455,0.852273,0.663017,0.602611,0.609006
70,0.155600,0.180133,0.534091,0.670455,0.806818,0.829545,0.534091,0.223485,0.161364,0.082955,0.534091,0.670455,0.806818,0.829545,0.677131,0.628075,0.636078
80,0.216700,0.208797,0.522727,0.681818,0.806818,0.818182,0.522727,0.227273,0.161364,0.081818,0.522727,0.681818,0.806818,0.818182,0.669806,0.621510,0.629680
90,0.116100,0.180733,0.511364,0.681818,0.806818,0.840909,0.511364,0.227273,0.161364,0.084091,0.511364,0.681818,0.806818,0.840909,0.673573,0.619634,0.626731
100,0.134200,0.200785,0.511364,0.681818,0.806818,0.840909,0.511364,0.227273,0.161364,0.084091,0.511364,0.681818,0.806818,0.840909,0.673737,0.619792,0.626444


TrainOutput(global_step=105, training_loss=0.4190982665334429, metrics={'train_runtime': 212.519, 'train_samples_per_second': 55.407, 'train_steps_per_second': 0.494, 'total_flos': 0.0, 'train_loss': 0.4190982665334429, 'epoch': 15.0})

In [ ]:
dev_evaluator(model)

{'lemma-retreiever_cosine_accuracy@1': 0.5227272727272727,
 'lemma-retreiever_cosine_accuracy@3': 0.6818181818181818,
 'lemma-retreiever_cosine_accuracy@5': 0.8068181818181818,
 'lemma-retreiever_cosine_accuracy@10': 0.8409090909090909,
 'lemma-retreiever_cosine_precision@1': 0.5227272727272727,
 'lemma-retreiever_cosine_precision@3': 0.22727272727272727,
 'lemma-retreiever_cosine_precision@5': 0.16136363636363635,
 'lemma-retreiever_cosine_precision@10': 0.08409090909090908,
 'lemma-retreiever_cosine_recall@1': 0.5227272727272727,
 'lemma-retreiever_cosine_recall@3': 0.6818181818181818,
 'lemma-retreiever_cosine_recall@5': 0.8068181818181818,
 'lemma-retreiever_cosine_recall@10': 0.8409090909090909,
 'lemma-retreiever_cosine_ndcg@10': 0.6777674394862633,
 'lemma-retreiever_cosine_mrr@10': 0.6253156565656566,
 'lemma-retreiever_cosine_map@100': 0.6319688719419088}

In [ ]:
10752 in lemmas_scopes

In [ ]:
list(lemmas_scopes.keys())[10_752]

In [ ]:
sorting_ids = set(df["positive_id"].astype(int))

In [ ]:
sorting_queries = {int(id): sequent for id, sequent in enumerate(ds["test"]["anchor"])}
sorting_corpus = {int(id): scope for id, scope in lemmas_scopes.items() if int(id) in sorting_ids}
sorting_relevant_docs = {qid: {lemmas_ids.get(sample["lemma_name"], 0)}
                 for qid, sample in enumerate(ds["test"])}


sorting_evaluator = InformationRetrievalEvaluator(
    queries=sorting_queries,
    corpus=sorting_corpus,{'lemma-retreiever_cosine_accuracy@1': 0.6136363636363636,
 'lemma-retreiever_cosine_accuracy@3': 0.875,
 'lemma-retreiever_cosine_accuracy@5': 0.9545454545454546,
 'lemma-retreiever_cosine_accuracy@10': 0.9772727272727273,
 'lemma-retreiever_cosine_precision@1': 0.6136363636363636,
 'lemma-retreiever_cosine_precision@3': 0.2916666666666667,
 'lemma-retreiever_cosine_precision@5': 0.1909090909090909,
 'lemma-retreiever_cosine_precision@10': 0.0977272727272727,
 'lemma-retreiever_cosine_recall@1': 0.6136363636363636,
 'lemma-retreiever_cosine_recall@3': 0.875,
 'lemma-retreiever_cosine_recall@5': 0.9545454545454546,
 'lemma-retreiever_cosine_recall@10': 0.9772727272727273,
 'lemma-retreiever_cosine_ndcg@10': 0.809018596884922,
 'lemma-retreiever_cosine_mrr@10': 0.753125,
 'lemma-retreiever_cosine_map@100': 0.7541597376201034}
    relevant_docs=sorting_relevant_docs,
    name="lemma-retreiever",
)

sorting_evaluator(model)

{'lemma-retreiever_cosine_accuracy@1': 0.6136363636363636,
 'lemma-retreiever_cosine_accuracy@3': 0.875,
 'lemma-retreiever_cosine_accuracy@5': 0.9545454545454546,
 'lemma-retreiever_cosine_accuracy@10': 0.9772727272727273,
 'lemma-retreiever_cosine_precision@1': 0.6136363636363636,
 'lemma-retreiever_cosine_precision@3': 0.2916666666666667,
 'lemma-retreiever_cosine_precision@5': 0.1909090909090909,
 'lemma-retreiever_cosine_precision@10': 0.0977272727272727,
 'lemma-retreiever_cosine_recall@1': 0.6136363636363636,
 'lemma-retreiever_cosine_recall@3': 0.875,
 'lemma-retreiever_cosine_recall@5': 0.9545454545454546,
 'lemma-retreiever_cosine_recall@10': 0.9772727272727273,
 'lemma-retreiever_cosine_ndcg@10': 0.809018596884922,
 'lemma-retreiever_cosine_mrr@10': 0.753125,
 'lemma-retreiever_cosine_map@100': 0.7541597376201034}

In [ ]:
# equal
a = model.encode(ds["test"][0]["anchor"])
b = model.encode(ds["test"][0]["positive"])

a @ b.T

np.float32(0.8771827)

In [ ]:
# negative
a = model.encode(ds["test"][0]["anchor"])
b = model.encode(ds["test"][1]["positive"])

a @ b.T

np.float32(0.29991314)

In [ ]:
model.save_pretrained("pvs-lemma-retreiever")

In [ ]:
!tar -cvf "pvs-lemma-retreiever.tar.gz" "pvs-lemma-retreiever"

pvs-lemma-retreiever/
pvs-lemma-retreiever/1_Pooling/
pvs-lemma-retreiever/1_Pooling/config.json
pvs-lemma-retreiever/vocab.json
pvs-lemma-retreiever/modules.json
pvs-lemma-retreiever/model.safetensors
pvs-lemma-retreiever/README.md
pvs-lemma-retreiever/config.json
pvs-lemma-retreiever/tokenizer.json
pvs-lemma-retreiever/merges.txt
pvs-lemma-retreiever/tokenizer_config.json
pvs-lemma-retreiever/sentence_bert_config.json
pvs-lemma-retreiever/2_Normalize/
pvs-lemma-retreiever/special_tokens_map.json
pvs-lemma-retreiever/config_sentence_transformers.json
